# Lab 07 solution

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("../../data")
OUT = Path("output")
OUT.mkdir(exist_ok=True)

raw = pd.read_csv(DATA / "trade_raw_2019_2023.csv")
countries = pd.read_excel(DATA / "countries.xlsx")
wb = pd.read_csv(DATA / "wb_indicators_2015_2023.csv")   # real World Bank data

log = []

In [ ]:
print(raw.shape)
print(raw.dtypes)
print(raw.isna().sum())
print("Exact duplicates:", raw.duplicated().sum())
for col in ["Reporter", "Partner", "Flow", "Unit"]:
    print(raw[col].value_counts(), end="\n\n")
print(raw["Period"].sample(10, random_state=0).tolist())

**Problems found:** reporter names in several spellings and cases, with stray spaces; `Value` stored as text because of thousands separators and `n/a`; blank values; a few negative values; a few rows in `USD thousands`; four different date formats in `Period`; exact duplicate rows.

In [ ]:
df = raw.copy()
df.columns = df.columns.str.lower()
for col in df.select_dtypes("object"):
    df[col] = df[col].str.strip()

print("Unmatched before:", sorted(set(df["reporter"]) - set(countries["country_name"])))
NAME_MAP = {
    "USA": "United States", "U.S.": "United States", "united states": "United States",
    "UK": "United Kingdom", "U.K.": "United Kingdom",
    "Vietnam": "Viet Nam",
    "germany": "Germany", "Federal Republic of Germany": "Germany",
}
changed = df["reporter"].isin(NAME_MAP).sum()
df["reporter"] = df["reporter"].replace(NAME_MAP)
log.append(f"Standardised {changed} reporter names to the countries.xlsx spelling")

In [ ]:
# check
assert set(df["reporter"]) <= set(countries["country_name"])
assert df["reporter"].nunique() == 14
print("Task 2 OK")

In [ ]:
text = df["value"].astype(str).str.replace(",", "", regex=False)
df["value"] = pd.to_numeric(text, errors="coerce")
log.append(f"Converted value to numeric; {df['value'].isna().sum()} blank or n/a values became missing")

thousands = df["unit"] == "USD thousands"
df.loc[thousands, "value"] /= 1000
df.loc[thousands, "unit"] = "USD millions"
log.append(f"Converted {thousands.sum()} rows from USD thousands to USD millions")

neg = df["value"] < 0
df.loc[neg, "value"] = df.loc[neg, "value"].abs()
log.append(f"{neg.sum()} negative export values treated as sign errors and made positive "
           "(magnitudes are consistent with neighbouring years); flagged for source query")

In [ ]:
# check
assert df["value"].dtype == float
assert (df["unit"] == "USD millions").all()
assert not (df["value"] < 0).any()
print("Task 3 OK")

In [ ]:
FORMATS = ["%Y-%m-%d", "%d/%m/%Y", "%b %Y", "%Y"]

def parse_period(p):
    for f in FORMATS:
        try:
            return pd.to_datetime(p, format=f)
        except ValueError:
            continue
    return pd.NaT

df["year"] = df["period"].map(parse_period).dt.year.astype(int)
log.append("Parsed four period formats into an integer year")

In [ ]:
# check
assert df["year"].notna().all()
assert set(df["year"]) == {2019, 2020, 2021, 2022, 2023}
print("Task 4 OK")

In [ ]:
before = len(df)
df = df.drop_duplicates()
log.append(f"Removed {before - len(df)} exact duplicate rows")

key = ["reporter", "partner", "year"]
key_dups = df[df.duplicated(key, keep=False)].sort_values(key)
print(key_dups)
# These rows were copies whose name or date spelling differed, so they only
# became visible as duplicates after standardising. Keep the row with a value.
df = (df.sort_values("value", na_position="last")
        .drop_duplicates(key, keep="first"))
log.append(f"Resolved {len(key_dups)} rows sharing a key, keeping the non-missing value")

df["value_missing"] = df["value"].isna()
log.append(f"Flagged {df['value_missing'].sum()} rows with missing values (kept, not imputed)")

In [ ]:
# check
assert not df.duplicated(["reporter", "partner", "year"]).any()
assert "value_missing" in df.columns
print("Task 5 OK")

In [ ]:
df = df.merge(countries[["iso3", "country_name", "region", "income_group"]],
              left_on="reporter", right_on="country_name", how="left",
              validate="many_to_one").drop(columns="country_name")

world = df[df["partner"] == "World"].merge(
    wb[["iso3", "year", "gdp_usd", "exports_pct_gdp"]], on=["iso3", "year"], how="left")
world["exports_pct_gdp_calc"] = (world["value"] * 1e6 / world["gdp_usd"] * 100).round(1)
world[["reporter", "year", "exports_pct_gdp_calc", "exports_pct_gdp"]].head(10)

In [ ]:
REQUIRED = ["iso3", "reporter", "partner", "year", "value", "unit"]

def validate(df):
    problems = []
    missing_cols = [c for c in REQUIRED if c not in df.columns]
    if missing_cols:
        return [f"missing columns: {missing_cols}"]
    if (df["value"] < 0).any():
        problems.append("negative values")
    if df["iso3"].isna().any():
        problems.append("unmatched reporters")
    if df.duplicated(["iso3", "partner", "year"]).any():
        problems.append("duplicate keys")
    if not df["year"].between(2019, 2023).all():
        problems.append("years out of range")
    if (df["unit"] != "USD millions").any():
        problems.append("mixed units")
    return problems

print(validate(df) or "All checks passed")

df = df.sort_values(["iso3", "partner", "year"])
df.to_csv(OUT / "trade_clean.csv", index=False)
(OUT / "cleaning_log.txt").write_text("\n".join(log), encoding="utf-8")
print("\n".join(log))

In [ ]:
# check
assert validate(df) == []
assert (OUT / "trade_clean.csv").exists()
print("Task 7 OK")